In [1]:
import math
import re
from collections import Counter


def limpiar_y_tokenizar(texto):
    """Limpia el texto eliminando signos de puntuación y lo divide en palabras."""
    texto = texto.lower()
    # Mantiene letras, números y tildes, elimina puntuación básica
    palabras = re.findall(r"\b[a-záéíóúñ0-9]+\b", texto)
    return palabras


# 📄 CORPUS DE PRUEBA: 12 Textos sobre Turismo con diferentes longitudes y ruidos
corpus_datos = {
    "Doc 1": "El turismo arqueológico en Machu Picchu arqueológico arqueológico arqueológico atrae a miles de viajeros que buscan descubrir la historia inca.",
    "Doc 2": "Las ruinas arqueológicas de Kuélap muestran la herencia de la cultura Chachapoyas un destino clave para el turismo arqueológico e histórico.",
    "Doc 3": "Un guía experto en arqueología explica los misterios de las líneas de Nasca un sitio arqueológico único en el mundo del turismo cultural.",
    "Doc 4": "El turismo de aventura en el cañón del Colca ofrece canotaje caminatas extremas y aventura aventura aventura aventura aventura aventura aventura aventura aventura aventura escalada en roca.",
    "Doc 5": "Hacer trekking y montañismo en la Cordillera Blanca es la máxima experiencia de turismo de aventura y adrenalina en la naturaleza alta.",
    "Doc 6": "Las dunas de Ica son perfectas para el turismo de aventura con la práctica de sandboard.",
    "Doc 7": "El turismo de naturaleza en la selva de Tambopata permite observar aves exóticas caimanes y naturaleza.",
    "Doc 8": "Visitar el parque nacional Huascarán es ideal para los amantes del turismo de naturaleza.",
    "Doc 9": "La reserva nacional de Paracas combina el turismo de naturaleza marina.",
    "Doc 10": "El turismo vivencial en las islas del lago Titicaca permite compartir costumbres.",
    "Doc 11": "Vivir en una comunidad nativa de la amazonía es una experiencia de turismo vivencial profunda.",
    "Doc 12": "El turismo vivencial en el Valle Sagrado conecta a los viajeros.",
}

# 🛠️ PROCESAMIENTO GENERAL DEL CORPUS
# 1. Tokenizar todos los documentos
corpus_tokenizado = {
    id_doc: limpiar_y_tokenizar(texto) for id_doc, texto in corpus_datos.items()
}

# 2. Obtener el vocabulario global de todo el corpus
vocabulario_global = set(
    [palabra for doc in corpus_tokenizado.values() for palabra in doc]
)

# 3. Calcular los rangos estadísticos independientes para CADA documento
umbrales_documentos = {}
for id_doc, palabras in corpus_tokenizado.items():
    conteos = Counter(palabras)
    frecuencias = list(conteos.values())

    if len(frecuencias) > 0:
        # Media de apariciones en ESTE documento
        media = sum(frecuencias) / len(frecuencias)
        # Desviación estándar en ESTE documento
        varianza = sum((x - media) ** 2 for x in frecuencias) / len(
            frecuencias
        )
        desviacion = math.sqrt(varianza)

        # Establecer la banda (Umbral Inferior y Superior adaptados a su distribución)
        u_inf = media + (0.5 * desviacion)
        u_sup = media + (2.5 * desviacion)

        # Si el documento es muy corto, aseguramos un rango funcional base
        if u_inf >= u_sup or len(palabras) < 30:
            u_inf = 1.5  # Exige al menos 2 apariciones para ser relevante
            u_sup = 4.5  # Bloquea si aparece 5 o más en textos micro

        umbrales_documentos[id_doc] = {"inf": u_inf, "sup": u_sup}

# 4. Calcular el DF (Document Frequency) usando tu Filtro de Banda Estadístico
df_banda = {}
for termino in vocabulario_global:
    conteo_df = 0
    for id_doc, palabras in corpus_tokenizado.items():
        tf_absoluto = palabras.count(termino)

        # Verificar si el término cae en la banda de aceptación de ESTE documento específico
        u_inf = umbrales_documentos[id_doc]["inf"]
        u_sup = umbrales_documentos[id_doc]["sup"]

        if u_inf <= tf_absoluto <= u_sup:
            conteo_df += 1

    df_banda[termino] = conteo_df

# 5. Calcular el IDF definitivo del algoritmo BB-IDF
N = len(corpus_datos)
bb_idf = {}
for termino, df_val in df_banda.items():
    # Suavizado de Laplace (+1) para evitar divisiones por cero si ninguna banda lo aceptó
    bb_idf[termino] = math.log(1 + (N / (df_val + 1)))


# 🎯 FUNCIÓN DE CONSULTA DE PALABRA CLAVE
def obtener_palabra_clave(id_documento):
    id_documento = id_documento.strip()
    if id_documento not in corpus_tokenizado:
        return "⚠️ El documento no existe en el corpus. Intenta con 'Doc 1', 'Doc 2', etc."

    palabras_doc = corpus_tokenizado[id_documento]
    palabras_unicas = set(palabras_doc)

    mejor_score = -1
    palabra_clave = None

    print(
        f"\n📊 Estadísticas internas para {id_documento} ({len(palabras_doc)} palabras):"
    )
    print(
        f"   -> Rango del filtro de banda permitido: [{umbrales_documentos[id_documento]['inf']:.2f} a {umbrales_documentos[id_documento]['sup']:.2f}] repeticiones"
    )

    # Evaluar el score de cada palabra de este documento
    for termino in palabras_unicas:
        tf = palabras_doc.count(termino)

        # Si la palabra no pasó el filtro de banda de este doc en el cálculo global, su score local se anula
        u_inf = umbrales_documentos[id_documento]["inf"]
        u_sup = umbrales_documentos[id_documento]["sup"]

        if u_inf <= tf <= u_sup:
            score = tf * bb_idf[termino]
        else:
            score = 0.0  # Filtrada por ruido o por exceso de spam

        if score > mejor_score:
            mejor_score = score
            palabra_clave = termino

    return f"🏆 La palabra clave de {id_documento} es: '{palabra_clave}' (Score BB-IDF: {mejor_score:.4f})"


# 🚀 EJECUCIÓN INTERACTIVA EN CONSOLA
if __name__ == "__main__":
    print("=== Sistema de Extracción de Palabras Clave BB-IDF ===")
    print("Documentos disponibles: Doc 1, Doc 2, Doc 3, ..., Doc 12")

    # Ejemplo automático al arrancar para validar el comportamiento
    print(obtener_palabra_clave("Doc 1"))
    print(obtener_palabra_clave("Doc 4"))

    # Bucle para que el usuario consulte
    while True:
        entrada = input(
            "\nIngresa el ID del documento a consultar (o 'salir'): "
        )
        if entrada.lower() == "salir":
            break
        print(obtener_palabra_clave(entrada))

=== Sistema de Extracción de Palabras Clave BB-IDF ===
Documentos disponibles: Doc 1, Doc 2, Doc 3, ..., Doc 12

📊 Estadísticas internas para Doc 1 (20 palabras):
   -> Rango del filtro de banda permitido: [1.50 a 4.50] repeticiones
🏆 La palabra clave de Doc 1 es: 'arqueológico' (Score BB-IDF: 7.7836)

📊 Estadísticas internas para Doc 4 (27 palabras):
   -> Rango del filtro de banda permitido: [1.50 a 4.50] repeticiones
🏆 La palabra clave de Doc 4 es: 'el' (Score BB-IDF: 3.2189)

Ingresa el ID del documento a consultar (o 'salir'): Doc 7

📊 Estadísticas internas para Doc 7 (16 palabras):
   -> Rango del filtro de banda permitido: [1.50 a 4.50] repeticiones
🏆 La palabra clave de Doc 7 es: 'naturaleza' (Score BB-IDF: 3.8918)

Ingresa el ID del documento a consultar (o 'salir'): Doc 12

📊 Estadísticas internas para Doc 12 (11 palabras):
   -> Rango del filtro de banda permitido: [1.50 a 4.50] repeticiones
🏆 La palabra clave de Doc 12 es: 'el' (Score BB-IDF: 3.2189)

Ingresa el ID del docu